# Optimize CLASP Parameters

Download datasets first with notebook 00. This notebook defines parameters, runs PCA latent Bayesian optimization, with optional GPLVM refinement, and saves the best settings for downstream notebooks.


In [1]:
import os

# Set native thread/OpenMP controls before importing torch, sklearn, scanpy, or clasp.
# This avoids macOS kernel crashes caused by mixed Intel/LLVM OpenMP runtimes.
for name in (
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
):
    os.environ[name] = "1"
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("KMP_INIT_AT_FORK", "FALSE")

from clasp.notebook_utils import (
    dataset_config,
    make_compact_search_space,
    make_estimator,
    make_clasp_optimization_objective,
    optimization_search_space,
    preprocess_params,
    run_latent_bayesopt,
    save_best_optimization_result,
)


Restart the kernel before running this notebook after changing OpenMP settings. The `notebooks/sitecustomize.py` file sets the same native-library controls at kernel startup, before ipykernel and compiled numerical libraries are imported.

In [2]:
selected_dataset = "scib_immune_human_mouse"
random_state = 0
embedding_method = "umap"
embedding_epochs = 60
invalid_score = -1e9
run_gplvm_refinement = True

pca_bo_settings = {
    "n_initial": 6,
    "latent_dim": 3,
    "n_iterations": 12,
    "embedding_model": "pca",
    "acquisition": "ei",
    "batch_size": 1,
}

gplvm_bo_settings = {
    "n_initial": 6,
    "latent_dim": 3,
    "n_iterations": 12,
    "embedding_model": "gplvm",
    "acquisition": "ei",
    "batch_size": 1,
}

dataset = dataset_config(selected_dataset)
raw_estimator = make_estimator(dataset, n_components=100, random_state=random_state)
raw_adata = raw_estimator.to_data(dataset["input_path"])
raw_adata


AnnData object with n_obs × n_vars = 97861 × 8135
    obs: 'batch', 'chemistry', 'data_type', 'dpt_pseudotime', 'final_annotation', 'mt_frac', 'n_counts', 'n_genes', 'size_factors', 'species', 'study', 'tissue'
    layers: 'counts'

In [3]:
fixed_preprocess_params = {"max_cells": 1000}

base_preprocess_params = preprocess_params(dataset, n_top_genes=1500)

base_estimator_params = {
    "n_components": 60,
}

base_graph_params = {
    "n_neighbors": 15,
    "intra_fraction": 0.5,
    "n_inter_edges": 2,
    "metric": "euclidean",
    "assignment_quantile": 0.8,
    "hubness_correction": "csls",
    "hubness_k": 10,
    "rank_correction": True,
    "edge_weighting": "binary",
    "mutual_neighbors": False,
    "neighbor_mode": "distance",
    "symmetrize": True,
}


In [4]:
preprocess_search_space = {
    "n_top_genes": {"type": "int", "bounds": [500, 2000]},
}

estimator_search_space = {
    "n_components": {"type": "int", "bounds": [20, 100]},
}

graph_search_space = {
    "n_neighbors": {"type": "int", "bounds": [5, 35]},
    "intra_fraction": {"type": "float", "bounds": [0.2, 0.9]},
    "n_inter_edges": {"type": "int", "bounds": [1, 6]},
    "assignment_quantile": {"type": "float", "bounds": [0.1, 1.0]},
    "hubness_k": {"type": "int", "bounds": [3, 20]},
    "rank_correction": {"type": "categorical", "values": [False, True]},
    "edge_weighting": {"type": "categorical", "values": ["binary", "distance"]},
    "mutual_neighbors": {"type": "categorical", "values": [False, True]},
}

search_space = optimization_search_space(
    preprocess_search_space=preprocess_search_space,
    estimator_search_space=estimator_search_space,
    graph_search_space=graph_search_space,
)
search_space


{'n_top_genes': {'type': 'int', 'bounds': [500, 2000]},
 'n_components': {'type': 'int', 'bounds': [20, 100]},
 'n_neighbors': {'type': 'int', 'bounds': [5, 35]},
 'intra_fraction': {'type': 'float', 'bounds': [0.2, 0.9]},
 'n_inter_edges': {'type': 'int', 'bounds': [1, 6]},
 'assignment_quantile': {'type': 'float', 'bounds': [0.1, 1.0]},
 'hubness_k': {'type': 'int', 'bounds': [3, 20]},
 'rank_correction': {'type': 'categorical', 'values': [False, True]},
 'edge_weighting': {'type': 'categorical', 'values': ['binary', 'distance']},
 'mutual_neighbors': {'type': 'categorical', 'values': [False, True]}}

In [5]:
objective = make_clasp_optimization_objective(
    dataset=dataset,
    raw_adata=raw_adata,
    base_preprocess_params=base_preprocess_params,
    fixed_preprocess_params=fixed_preprocess_params,
    base_estimator_params=base_estimator_params,
    base_graph_params=base_graph_params,
    preprocess_search_space=preprocess_search_space,
    estimator_search_space=estimator_search_space,
    graph_search_space=graph_search_space,
    random_state=random_state,
    embedding_method=embedding_method,
    embedding_epochs=embedding_epochs,
    invalid_score=invalid_score,
)


In [6]:
pca_result = run_latent_bayesopt(
    objective,
    search_space,
    **pca_bo_settings,
    random_state=random_state,
    invalid_score=invalid_score,
)

pca_result["best_params"], pca_result["best_score"]


({'n_top_genes': 1106,
  'n_components': 58,
  'n_neighbors': 35,
  'intra_fraction': 0.59560809472893,
  'n_inter_edges': 6,
  'assignment_quantile': 1.0,
  'hubness_k': 10,
  'rank_correction': True,
  'edge_weighting': 'distance',
  'mutual_neighbors': False},
 0.7581)

In [7]:
pca_result["history"].sort_values("score", ascending=False).head(10)


,iteration,phase,score,n_top_genes,n_components,n_neighbors,intra_fraction,n_inter_edges,assignment_quantile,hubness_k,rank_correction,edge_weighting,mutual_neighbors
8,3,bo,0.75810,1106,58,35,0.595608,6,1.000000,10,True,distance,False
17,12,bo,0.74960,1276,59,35,0.506882,6,1.000000,12,True,distance,False
12,7,bo,0.74500,1372,68,35,0.557604,5,1.000000,14,True,distance,False
9,4,bo,0.74470,1167,53,35,0.462847,6,0.945490,10,True,distance,False
11,6,bo,0.74135,889,52,35,0.657857,6,1.000000,8,True,distance,False
13,8,bo,0.73840,1413,69,35,0.556947,5,1.000000,14,True,distance,False
14,9,bo,0.73550,1448,71,35,0.560816,5,1.000000,15,True,distance,False
6,1,bo,0.73460,1047,73,35,0.900000,5,1.000000,12,True,distance,False
15,10,bo,0.72850,939,52,35,0.632846,6,1.000000,9,True,distance,False
1,0,initial,0.72665,1255,69,35,0.580537,5,0.941565,7,True,distance,False


In [8]:
compact_radii = {
    "n_top_genes": 300,
    "n_components": 20,
    "n_neighbors": 6,
    "intra_fraction": 0.1,
    "n_inter_edges": 2,
    "assignment_quantile": 0.15,
    "hubness_k": 4,
}

compact_search_space = make_compact_search_space(
    search_space,
    pca_result["best_params"],
    compact_radii,
    fix_categoricals=True,
)
compact_search_space


{'n_top_genes': {'type': 'int', 'bounds': [806, 1406]},
 'n_components': {'type': 'int', 'bounds': [38, 78]},
 'n_neighbors': {'type': 'int', 'bounds': [29, 35]},
 'intra_fraction': {'type': 'float',
  'bounds': [0.49560809472892997, 0.6956080947289299]},
 'n_inter_edges': {'type': 'int', 'bounds': [4, 6]},
 'assignment_quantile': {'type': 'float', 'bounds': [0.85, 1.0]},
 'hubness_k': {'type': 'int', 'bounds': [6, 14]},
 'rank_correction': {'type': 'categorical', 'values': [True]},
 'edge_weighting': {'type': 'categorical', 'values': ['distance']},
 'mutual_neighbors': {'type': 'categorical', 'values': [False]}}

In [9]:
if run_gplvm_refinement:
    gplvm_result = run_latent_bayesopt(
        objective,
        compact_search_space,
        **gplvm_bo_settings,
        random_state=random_state + 1,
        invalid_score=invalid_score,
    )
    gplvm_summary = (gplvm_result["best_params"], gplvm_result["best_score"])
else:
    gplvm_result = None
    gplvm_summary = "skipped"

gplvm_summary


({'n_top_genes': 1279,
  'n_components': 43,
  'n_neighbors': 31,
  'intra_fraction': 0.5863076726250602,
  'n_inter_edges': 6,
  'assignment_quantile': 0.9104669479670694,
  'hubness_k': 7,
  'rank_correction': True,
  'edge_weighting': 'distance',
  'mutual_neighbors': False},
 0.75425)

In [10]:
if gplvm_result is not None:
    gplvm_result["history"].sort_values("score", ascending=False).head(10)


In [11]:
optimization_results = {"pca": pca_result}
if gplvm_result is not None:
    optimization_results["gplvm"] = gplvm_result

params_path, optimized_preprocess_params, optimized_estimator_params, optimized_graph_params = save_best_optimization_result(
    dataset_name=selected_dataset,
    optimization_results=optimization_results,
    base_preprocess_params=base_preprocess_params,
    fixed_preprocess_params=fixed_preprocess_params,
    base_estimator_params=base_estimator_params,
    base_graph_params=base_graph_params,
    preprocess_search_space=preprocess_search_space,
    estimator_search_space=estimator_search_space,
    graph_search_space=graph_search_space,
    random_state=random_state,
)

params_path, optimized_preprocess_params, optimized_estimator_params, optimized_graph_params


(PosixPath('/Users/fabriziocosta/Resilio Sync/Sync/Projects/ACTIVE/clasp/data/optimized_params/scib_immune_human_mouse_graph_params.json'),
 {'n_top_genes': 1106,
  'max_cells': 1000,
  'min_cell_genes': None,
  'min_gene_counts': 0,
  'normalize': False,
  'hvg_flavor': 'variance',
  'create_artificial_batch': False},
 {'n_components': 58},
 {'n_neighbors': 35,
  'intra_fraction': 0.59560809472893,
  'n_inter_edges': 6,
  'metric': 'euclidean',
  'assignment_quantile': 1.0,
  'hubness_correction': 'csls',
  'hubness_k': 10,
  'rank_correction': True,
  'edge_weighting': 'distance',
  'mutual_neighbors': False,
  'neighbor_mode': 'distance',
  'symmetrize': True})